# 09 — Hybrid BM25 + Vector Search

AI-Lake provides first-class **hybrid lexical+vector search** (Fase 5) without external search engines:

| API | Description |
|---|---|
| `TableWriter(bm25_text_column=…)` | Accumulate BM25 IDF stats at write time |
| `ailake.search_text(path, query, …)` | Pure BM25 brute-force scan — no embedding required |
| `ailake.search(…, hybrid_text=…)` | Hybrid: HNSW + BM25 fused via RRF or linear combination |

BM25 stats are stored as `metadata/ailake_bm25_stats.bin` (zstd-compressed bincode),
keyed in Iceberg table properties as `ailake.bm25.stats-path`.

**When to use which:**
- `search_text` — keyword lookups, O(N), no embedding required, works offline.
- `search` with `hybrid_text` — best precision when both semantic and keyword signals matter (RAG pipelines, product search).
- `search` without `hybrid_text` — pure HNSW for semantic similarity (default).

In [ ]:
import json, math, pathlib, numpy as np
import ailake

# Load demo fixture paths
PAYLOAD_PATH = pathlib.Path('/data/demo_query.json')
if PAYLOAD_PATH.exists():
    payload = json.loads(PAYLOAD_PATH.read_text())
    BM25_PATH      = payload['table_paths']['bm25']
    HNSW_PATH      = payload['table_paths']['hnsw']
    QUERY_VEC      = payload['query_vector']
    DIM            = payload['dim']
    METRIC         = payload['metric']
else:
    # Fallback: create a minimal fixture inline
    BM25_PATH = '/tmp/nb09_bm25'
    HNSW_PATH = '/tmp/nb09_hnsw'
    DIM, METRIC = 32, 'cosine'

print(f'BM25 table: {BM25_PATH}')
print(f'DIM={DIM}  METRIC={METRIC}')

## 1. Write a table with BM25 indexing enabled

Pass `bm25_text_column='text'` to `TableWriter`. AI-Lake will:
1. Accumulate **document-frequency (DF) counts** from the named column on every `write_batch`.
2. Compute **IDF scores** (BM25+ formula) from DF counts.
3. Persist stats to `metadata/ailake_bm25_stats.bin`.

The same table is fully searchable via HNSW — BM25 indexing is additive.

In [ ]:
import pathlib

NB_BM25_PATH = '/tmp/nb09_bm25_demo'
pathlib.Path(NB_BM25_PATH).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

topics = [
    'rust programming systems language memory safety ownership',
    'python machine learning data science numpy pandas',
    'rust async tokio concurrency futures channels',
    'javascript typescript react frontend components',
    'rust memory borrowing lifetimes references',
    'database sql query optimization index btree',
    'vector search approximate nearest neighbor hnsw',
    'distributed computing apache spark hadoop mapreduce',
    'rust cargo crates dependencies ecosystem',
    'deep learning neural network transformer attention',
]

# Synthetic embeddings — different random vectors per topic
embeddings = rng.standard_normal((len(topics), DIM)).astype(np.float32)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)

w = ailake.TableWriter(NB_BM25_PATH, dim=DIM, metric=METRIC, bm25_text_column='text')
w.write_batch(topics, embeddings.tolist())
snap = w.commit()
print(f'Table written: {len(topics)} rows, snapshot_id={snap}')
print(f'BM25 stats stored at: {NB_BM25_PATH}/metadata/ailake_bm25_stats.bin')

## 2. Pure BM25 text search (`search_text`)

`ailake.search_text` does a brute-force BM25 scan — **no embedding required**.
Results are ranked by BM25 score (lower `distance` = more relevant, same convention as HNSW).

In [ ]:
import ailake

query = 'rust concurrency async programming'

text_results = ailake.search_text(
    NB_BM25_PATH,
    query_text=query,
    top_k=5,
    text_column='text',
)

print(f'Query: "{query}"\n')
print(f'Top-{len(text_results)} BM25 results:')
for i, r in enumerate(text_results, 1):
    print(f'  [{i}] row_id={r["row_id"]:2d}  score={-r["distance"]:6.4f}  text="{topics[r["row_id"]][:60]}"')

## 3. Hybrid search: vector + BM25 via RRF

Pass `hybrid_text=query` to `ailake.search` to enable hybrid mode.

**Pipeline:**
1. HNSW retrieves `candidate_pool` vectors (default `10 × top_k`).
2. BM25 scores each candidate against `hybrid_text`.
3. Results are **fused via Reciprocal Rank Fusion (RRF)**:
   `score = (1 - bm25_weight) / (60 + rank_vec) + bm25_weight / (60 + rank_bm25)`

Lower fused score = ranked higher.

In [ ]:
# Use the embedding of the first rust+async topic as the query vector
query_vec = embeddings[2].tolist()  # 'rust async tokio concurrency…'
query_text = 'rust concurrency async programming'

# Pure vector search (baseline) — materialise via .to_list()
vec_results    = ailake.search(NB_BM25_PATH, query_vec, top_k=5).to_list()

# Hybrid: 60% BM25 / 40% vector weight
hybrid_results = ailake.search(
    NB_BM25_PATH,
    query_vec,
    top_k=5,
    hybrid_text=query_text,
    text_column='text',
    bm25_weight=0.6,
).to_list()

import pandas as pd
print('Pure vector search:')
for i, r in enumerate(vec_results, 1):
    print(f'  [{i}] row_id={r["row_id"]:2d}  dist={r["distance"]:.4f}  "{topics[r["row_id"]][:55]}"')

print(f'\nHybrid (BM25 weight=0.6, fusion=RRF):')
for i, r in enumerate(hybrid_results, 1):
    print(f'  [{i}] row_id={r["row_id"]:2d}  rrf={-r["distance"]:6.4f}  "{topics[r["row_id"]][:55]}"')

## 4. Weight ablation: pure vector vs. hybrid vs. pure BM25

Adjust `bm25_weight` between 0.0 (pure vector) and 1.0 (pure BM25).

In [ ]:
import pandas as pd

weights = [0.0, 0.3, 0.5, 0.7, 1.0]
rows = []
for w in weights:
    if w == 0.0:
        results = ailake.search(NB_BM25_PATH, query_vec, top_k=3).to_list()
    elif w == 1.0:
        # Pure text search via search_text (already returns a list)
        results = ailake.search_text(NB_BM25_PATH, query_text, top_k=3, text_column='text')
    else:
        results = ailake.search(NB_BM25_PATH, query_vec, top_k=3,
                                hybrid_text=query_text, text_column='text', bm25_weight=w).to_list()
    top3_ids = [r['row_id'] for r in results]
    rows.append({'bm25_weight': w, 'top3_row_ids': top3_ids})

df = pd.DataFrame(rows)
print(f'Query: "{query_text}"\n')
print(df.to_string(index=False))

## 5. Use the pre-built BM25 fixture table

The demo fixture at `BM25_PATH` was written by `init_demo.py` with `bm25_text_column='text'`.

In [ ]:
if PAYLOAD_PATH.exists():
    fixture_text_results = ailake.search_text(
        BM25_PATH,
        query_text='vector search approximate nearest neighbor',
        top_k=5,
        text_column='text',
    )
    print(f'Fixture BM25 table: {len(fixture_text_results)} results')
    for r in fixture_text_results:
        print(f'  row_id={r["row_id"]:3d}  bm25={-r["distance"]:6.3f}  file={r["file"][-30:]}')
else:
    print('Demo fixture not available — run init_demo.py first.')

## 6. `WorkingMemoryBuffer` — agent short-term memory

`ailake.WorkingMemoryBuffer` is a bounded in-memory queue for agent short-term context.
When full, the oldest entry is evicted (FIFO). Supports brute-force cosine search and draining to an AI-Lake table.

In [ ]:
wm = ailake.WorkingMemoryBuffer(max_rows=5)

# Push 6 entries — buffer evicts the oldest when capacity (5) is exceeded
for i, text in enumerate(topics[:6]):
    emb = embeddings[i].tolist()
    wm.push(text, emb, importance=0.8 if 'rust' in text else 0.4)
    print(f'Pushed [{i}]: len={len(wm)}  full={wm.is_full()}')

print(f'\nBuffer has {len(wm)} entries (oldest evicted)')

In [ ]:
# Brute-force cosine search in the buffer
query_emb = embeddings[2].tolist()  # rust async tokio
hits = wm.search(query_emb, top_k=3)
print('Top-3 in working memory:')
for h in hits:
    print(f'  dist={h["distance"]:.4f}  importance={h["importance"]:.1f}  "{h["text"][:60]}"')

In [ ]:
# Drain buffer to a persistent AI-Lake table
WM_TABLE_PATH = '/tmp/nb09_wm_table'
pathlib.Path(WM_TABLE_PATH).mkdir(parents=True, exist_ok=True)

drain_writer = ailake.TableWriter(WM_TABLE_PATH, dim=DIM, metric=METRIC)
wm.drain_to_table(drain_writer)
snap = drain_writer.commit()
print(f'Buffer drained to {WM_TABLE_PATH}  snapshot_id={snap}')
print(f'Buffer empty after drain: {wm.is_empty()}')

## 7. `decay_memories` — periodic recency decay

`ailake.decay_memories(path, decay_lambda)` reads the `last_accessed_at` column
from each data file and recomputes `recency_weight = exp(-lambda * days_since_access)`.

**When to use:** Call periodically (e.g. nightly) on episodic memory tables to ensure
stale memories are naturally down-ranked in hybrid scoring.

In [ ]:
import datetime, ailake

DECAY_TABLE_PATH = '/tmp/nb09_decay_demo'
pathlib.Path(DECAY_TABLE_PATH).mkdir(parents=True, exist_ok=True)

now = datetime.datetime.now(datetime.UTC)
texts_d  = ['Recent memory: today', 'Week-old memory', 'Month-old memory']
ages_d   = [0, 7, 30]  # days old
embs_d   = rng.standard_normal((3, DIM)).astype(np.float32)
embs_d  /= np.linalg.norm(embs_d, axis=1, keepdims=True)

# Write with extra columns (last_accessed_at, recency_weight)
w = ailake.TableWriter(DECAY_TABLE_PATH, dim=DIM, metric=METRIC)
w.write_batch(
    texts_d,
    embs_d.tolist(),
    extra_columns={
        'last_accessed_at': [
            (now - datetime.timedelta(days=d)).strftime('%Y-%m-%dT%H:%M:%S')
            for d in ages_d
        ],
        'recency_weight': [1.0] * 3,  # initial weights (will be recomputed)
    },
)
w.commit()

# Run decay job: lambda=0.1 → half-life ≈ 7 days
updated = ailake.decay_memories(DECAY_TABLE_PATH, decay_lambda=0.1)
print(f'Files updated by decay job: {updated}')

# Expected weights after decay
for text, days in zip(texts_d, ages_d):
    rw = math.exp(-0.1 * days)
    print(f'  days_old={days:2d}  recency_weight={rw:.3f}  "{text}"')